In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    cross_val_score, learning_curve, validation_curve, KFold, TimeSeriesSplit
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# =============================================================
# PARTIE 1 : VISUALISER LE BIAIS-VARIANCE
# =============================================================
print("=" * 60)
print("PARTIE 1 : Visualisation Biais-Variance")
print("=" * 60)

# Générer des données synthétiques : y = sin(2πx) + bruit
np.random.seed(42)
n_samples = 50
X = np.sort(np.random.uniform(0, 1, n_samples))
y_true = np.sin(2 * np.pi * X)  # La vraie fonction (inconnue en pratique)
y = y_true + np.random.normal(0, 0.3, n_samples)  # Avec bruit

X = X.reshape(-1, 1)
X_plot = np.linspace(0, 1, 200).reshape(-1, 1)

# Fitter des polynômes de différents degrés
degrees = [1, 4, 15]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, degree in zip(axes, degrees):
    # Créer un pipeline : PolynomialFeatures + LinearRegression
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X, y)
    
    # Prédictions
    y_pred_plot = model.predict(X_plot)
    y_pred_train = model.predict(X)
    
    # Erreur sur le train
    mse_train = mean_squared_error(y, y_pred_train)
    
    # Tracer
    ax.scatter(X, y, color='blue', alpha=0.5, label='Données (avec bruit)')
    ax.plot(X_plot, np.sin(2 * np.pi * X_plot), 'g--', label='Vraie fonction')
    ax.plot(X_plot, y_pred_plot, 'r-', linewidth=2, label=f'Polynôme degré {degree}')
    ax.set_title(f'Degré {degree}\nMSE train = {mse_train:.4f}')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=8)

plt.suptitle("Le dilemme Biais-Variance : 3 degrés de complexité", fontsize=14)
plt.tight_layout()
plt.savefig("biais_variance_demo.png", dpi=150)
plt.show()

# TODO : Répondez aux questions suivantes :
# 1. Quel modèle a la plus faible erreur d'entraînement ?
# 2. Quel modèle généralisera le mieux à de nouvelles données ? Pourquoi ?
# 3. Le modèle de degré 15 a une MSE train presque nulle. Est-ce bon signe ?

# =============================================================
# PARTIE 2 : COURBES D'APPRENTISSAGE
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 2 : Courbes d'apprentissage")
print("=" * 60)

# Générer plus de données pour les learning curves
np.random.seed(42)
n_samples_lc = 300
X_lc = np.sort(np.random.uniform(0, 1, n_samples_lc)).reshape(-1, 1)
y_lc = np.sin(2 * np.pi * X_lc.ravel()) + np.random.normal(0, 0.3, n_samples_lc)

# Comparer deux modèles : un biaisé et un à haute variance
models = {
    'Polynôme degré 1 (biais élevé)': make_pipeline(PolynomialFeatures(1), LinearRegression()),
    'Polynôme degré 15 (variance élevée)': make_pipeline(PolynomialFeatures(15), LinearRegression()),
    'Polynôme degré 4 (bon compromis)': make_pipeline(PolynomialFeatures(4), LinearRegression()),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_lc, y_lc, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, 
        scoring='neg_mean_squared_error',
        random_state=42
    )
    
    # Convertir en erreur positive
    train_errors = -train_scores.mean(axis=1)
    val_errors = -val_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.plot(train_sizes, train_errors, 'b-', label='Erreur Train')
    ax.fill_between(train_sizes, train_errors - train_std, train_errors + train_std, alpha=0.1, color='blue')
    ax.plot(train_sizes, val_errors, 'r-', label='Erreur Validation')
    ax.fill_between(train_sizes, val_errors - val_std, val_errors + val_std, alpha=0.1, color='red')
    ax.set_xlabel('Nombre d\'exemples d\'entraînement')
    ax.set_ylabel('MSE')
    ax.set_title(name)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.set_xlim(10, n_samples_lc)